# പാഠം 11 - ഏജന്റ്-ടു-ഈജന്റ് (A2A) പ്രോട്ടോക്കോൾ


## സജ്ജീകരണം


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## What is the A2A Protocol?

The **Agent-to-Agent (A2A) protocol** is an open standard that enables AI agents to communicate,
discover each other, and collaborate — even when they are built on different frameworks or hosted
by different services.

Key concepts:

- **Discovery** – Agents publish an *Agent Card* that describes their capabilities, making it
  easy for other agents (or orchestrators) to find the right specialist for a task.
- **Message Passing** – Agents exchange structured messages through a common protocol, so a
  request from one agent can be understood and fulfilled by another regardless of its internal
  implementation.
- **Task Lifecycle** – A2A defines states such as *submitted*, *working*, *completed*, and
  *failed*, giving the orchestrator full visibility into how a delegated task is progressing.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## പ്രത്യേക ട്രാവൽ ഏജന്റുമാർ സൃഷ്ടിക്കൽ


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## വർക്ക്‌ഫ്ലോ വഴി ബഹുഏജൻറ് സഹകരണങ്ങൾ

നാം മൂന്ന് ഏജൻറുകളെയും A2A സന്ദേശം കൈമാറുന്നതിന്റെ അനുകരണമായ ക്രമാനുസൃത വർക്ക്‌ഫ്ലോയിലേക്ക് ബന്ധിപ്പിക്കുന്നു:

1. **CurrencyExchangeAgent** ഉപയോക്തൃ അഭ്യർത്ഥന സ്വീകരിച്ച് കറൻസി നിർദ്ദേശം നൽകുന്നു.
2. **ActivityPlannerAgent** സമ്പുഷ്ടമായ സാന്റർഭം സ്വീകരിച്ച് പ്രവർത്തന നിർദ്ദേശങ്ങൾ ചേർക്കുന്നു.
3. **TravelManagerAgent** ഇരുവിധ/input വായനകൾ സംയോജിപ്പിച്ച് അന്തിമ യാത്രാ സൂചന തയ്യാറാക്കുന്നു.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## പ്രൊഡക്ഷനിൽ A2A മനസിലാക്കുക

പ്രൊഡക്ഷൻ പരിസരത്ത് A2A പ്രോട്ടോക്കോൾ ശക്തമായ ക്രോസ്-സേവന സെനാരിയോകൾ അന്ലോക്ക് ചെയ്യുന്നു:

| കഴിവ് | വിവരണം |
|---|---|
| **ക്രോസ്-ഫ്രെയിമ്വർക്കിന്റെ ഇടപഴകൽ** | ഒരു ഫ്രെയിംവർക്ക് ഉപയോഗിച്ച് നിർമ്മിച്ച ഒരു ഏജന്റ് മറ്റേതെങ്കിലും A2A-അനുമതി ലഭ്യമായ ഫ്രെയിംവർക്കിൽ നിർമ്മിച്ച ഏജന്റിന് തസ്തികകൾ കൈമാറാൻ കഴിയും, ഇത് യഥാർത്ഥ ക്രോസ്-സംഘടന ഇടപഴകலை സജീവമാക്കുന്നു. |
| **സേവന പരിധികൾ** | ഏജന്റുകൾ വേർതിരിച്ച മൈക്രോസ്‌വർമിസസുകളിലും ക്ലൗഡ് റീജിയങ്ങളിലും വ്യത്യസ്ത സ്ഥാപനങ്ങളിലുമെല്ലാം ജീവിക്കാൻ കഴിയും, ഇതുവരെulem സുതാര്യമായ സഹകരണവും സാധൂകരിക്കുന്നു. |
| **ഡൈനാമിക് ഡിസ്കവറി** | ഒരു ഓർക്കസ്ട്രേറ്റർ റൺടൈമിൽ ഏജന്റ് കാർഡ് രജിസ്ട്രിയിൽ ചോദിച്ച് ഒരു നിശ്ചിത ഉപ-തസ്തികക്ക് ഏറ്റവും അനുയോജ്യനായ വിദഗ്ദ്ധനെ കണ്ടെത്തും. |
| **സ്റ്റ്രീമിംഗ് & പുഷ് അറിയിപ്പുകൾ** | A2A യഥാർത്ഥ സമയ പുരോഗതി അപ്‌ഡേറ്റുകൾക്കും ദീർഘകാല പ്രവർത്തനത്തിനുള്ള പുഷ് അറിയിപ്പുകൾക്കും സെർവർ-സന്റോണ്റഡ് ഇവന്റുകൾ (SSE) പിന്തുണയ്ക്കുന്നു. |

മുകളിൽ നിർമിച്ച വർക്ക്ഫ്‌ളോ ഈ പാറ്റേണിന്റെ ലളിതമാക്കിയ, ഇൻ-പ്രോസസ് പതിപ്പാണ്. യഥാർത്ഥ
വിന്യാസത്തിൽ ഓരോ ഏജന്റും ഒരു HTTP എന്ദ്പോയിന്റ് പുറപ്പെടുവിക്കുകയും, ഏജന്റ് കാർഡ് പ്രസിദ്ധീകരിക്കുകയും
A2A JSON-RPC പ്രോട്ടോക്കോൾ വഴി ആശയവിനിമയം നടത്തുകയും ചെയ്യും.


## സംഗ്രഹം

ഈ പാഠത്തില്‍ നിങ്ങള്‍ പഠിച്ചത്:

1. **A2A പ്രോട്ടോകോള്‍ എന്താണ്** — ഏജന്റ്-ടു-ഏജന്റ് കണ്ടെത്തല്‍, സന്ദേശവിനിമയം, 
   കൂടാതെ ടാസ്‌ക് മാനേജ്മെന്റിനായുള്ള ഒരു തുറന്ന മാനദണ്ഡം.
2. **വിശേഷിപ്പിച്ച ഏജന്റുമാര്‍ എങ്ങനെ സൃഷ്ടിക്കാം** — ഒരു കറന്‍സി എക്‌സ്‌ചേഞ്ച് ഏജന്റ്, ഒരു ആക്റ്റിവിറ്റി പ്ലാനര്‍ ഏജന്റ്, 
   കൂടാതെ ഒരു ട്രാവല്‍ മാനേജര്‍ ഓർക്കസ്ട്രേറ്റര്‍.
3. **ഏജന്റുമാരെ ഒരു വര്‍ക്ക്ഫ്ലോയില്‍ എങ്ങനെ കണക്ട് ചെയ്യണം** — ഏജന്റുമാരുടെ ഇടയിലുള്ള ക്രമബദ്ധ സന്ദേശ സംപ്രേഷണങ്ങള്‍
   മോഡലാക്കാന്‍ `WorkflowBuilder` ഉപയോഗമുള്ള രീതി.
4. **ഉത്പാദനത്തില്‍ A2A എങ്ങനെ പ്രവർത്തിക്കുന്നു** — ഡൈനാമിക് കണ്ടെത്തലും സ്ട്രീമിംഗ് അപ്‌ഡേറ്റുകളും ഉപയോഗിച്ച് 
   ക്രോസ്-ഫ്രെയിംവര്‍ക്ക്, ക്രോസ്-സെർവീസ് സഹകരണത്തിന് പ്രോത്സാഹനം നൽകുന്നു.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
